# Customer 360 View
## Project Execution Guidelines

### Step 1: Data Loading and Initial Exploration

* Load all datasets using Pandas
* Inspect structure using `.head()`, `.info()`, `.describe()`
* Identify primary and foreign keys
* Understand relationships between tables

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob
#!pip install pathlib
from pathlib import Path
current_dir = Path.cwd()
data_folder = current_dir/"python_project_aiml_logicmojo_dataset"
#print(data_folder)
data_files = glob.glob(f"{data_folder}/*.csv")
dataframes = {}
#Load all datasets using Pandas
for file_path_str in data_files:
    file_path = Path(file_path_str)
    #print(file_path)
    key = file_path.stem
    try:
        dataframes[key] = pd.read_csv(file_path)
        # Inspect structure using
        print(f"\nFirst {key} 3 rows:")
        print(dataframes[key].head(3))
        print(f"\nInformation about {key}:")
        print(dataframes[key].info())
        print(f"\nInformation about {key} Shape:")
        print(dataframes[key].shape)
        print(f"\n {key} Summary Statistics:")
        print(dataframes[key].describe())
        print(f"\n  {key} Missing Values:")
        print(dataframes[key].isnull().sum())
        print(f"\n  {key} Duplicate Values:")
        print(dataframes[key].duplicated().sum())
        
    except Exception as e:
        print(f"Failed to load {var_name} : {e}")


First customers 3 rows:
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

Information about customers:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3

In [2]:
all_column_lists = []
for df in dataframes.values():
    all_column_lists.append(list(df.columns))
#print(all_column_lists)
all_column_sets = []
for col_list in all_column_lists:
    all_column_sets.append(set(col_list))
print(all_column_sets)

[{'customer_city', 'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_state'}, {'product_name_lenght', 'product_id', 'product_height_cm', 'product_category_name', 'product_weight_g', 'product_width_cm', 'product_photos_qty', 'product_length_cm', 'product_description_lenght'}, {'review_creation_date', 'review_comment_message', 'review_comment_title', 'review_score', 'order_id', 'review_id', 'review_answer_timestamp'}, {'customer_id', 'order_purchase_timestamp', 'order_status', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_id', 'order_approved_at'}, {'product_category_name', 'product_category_name_english'}, {'geolocation_state', 'geolocation_lng', 'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_city'}, {'payment_value', 'payment_sequential', 'payment_type', 'order_id', 'payment_installments'}, {'seller_city', 'seller_id', 'seller_state', 'seller_zip_code_prefix'}, {'product_id', 'seller_id'

In [3]:
# Manually assigning each table to its own explicit variable
customers            = dataframes['customers'].copy()
products             = dataframes['products'].copy()
reviews              = dataframes['reviews'].copy()
orders               = dataframes['orders'].copy()
category_translation = dataframes['category_translation'].copy()
location             = dataframes['location'].copy()
payments             = dataframes['payments'].copy()
sellers              = dataframes['sellers'].copy()
order_item           = dataframes['order_item'].copy()


### Step 2: Data Cleaning and Preprocessing

* Handle missing values appropriately
* Remove duplicate records
* Convert date columns to datetime format
* Validate data types and ranges
* Standardize column names if required

In [4]:
df_to_clean = {
    'customers': customers, 'products': products, 'reviews': reviews, 
    'orders': orders, 'category_translation': category_translation, 
    'location': location, 'payments': payments, 'sellers': sellers, 
    'order_item': order_item
}
#Making sure column names are consistent (assign back so it actually takes effect)
for name, df in df_to_clean.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
print("Column standardisation complete. Sample:")
for name, df in df_to_clean.items():
    print(f"  {name}: {list(df.columns[:3])}...")

Column standardisation complete. Sample:
  customers: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix']...
  products: ['product_id', 'product_category_name', 'product_name_lenght']...
  reviews: ['review_id', 'order_id', 'review_score']...
  orders: ['order_id', 'customer_id', 'order_status']...
  category_translation: ['product_category_name', 'product_category_name_english']...
  location: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng']...
  payments: ['order_id', 'payment_sequential', 'payment_type']...
  sellers: ['seller_id', 'seller_zip_code_prefix', 'seller_city']...
  order_item: ['order_id', 'order_item_id', 'product_id']...


In [5]:
#Remove duplicate records
for name, df in df_to_clean.items():
    initial_shape = df.shape
    dupli = df.duplicated().sum()
    if dupli > 0:
        df.drop_duplicates(inplace=True)
        print(f"{name}: Removed {dupli} duplicates. Old Shape {initial_shape}, new shape {df.shape}")
    else:
        print(f"{name}: Zero duplicates found. Shape {df.shape}")


customers: Zero duplicates found. Shape (99441, 5)
products: Zero duplicates found. Shape (32951, 9)
reviews: Zero duplicates found. Shape (99224, 7)
orders: Zero duplicates found. Shape (99441, 8)
category_translation: Zero duplicates found. Shape (71, 2)
location: Removed 261831 duplicates. Old Shape (1000163, 5), new shape (738332, 5)
payments: Zero duplicates found. Shape (103886, 5)
sellers: Zero duplicates found. Shape (3095, 4)
order_item: Zero duplicates found. Shape (112650, 7)


In [6]:
#Fix the date format - convert each column from its OWN data (not from another table)

#Reviews Table
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

#Order Table - each date column converts itself
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'], errors='coerce')
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'], errors='coerce')
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'], errors='coerce')
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'], errors='coerce')
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'], errors='coerce')

#Order Item Table
order_item['shipping_limit_date'] = pd.to_datetime(order_item['shipping_limit_date'], format='mixed', errors='coerce')

#Verify the conversions
print("Date columns verified:")
print(orders[['order_purchase_timestamp', 'order_delivered_customer_date']].dtypes)
print(f"\nSample order dates (should be different from each other):")
print(orders[['order_purchase_timestamp', 'order_delivered_customer_date']].head(3))

Date columns verified:
order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
dtype: object

Sample order dates (should be different from each other):
  order_purchase_timestamp order_delivered_customer_date
0      2017-10-02 10:56:33           2017-10-10 21:25:13
1      2018-07-24 20:41:37           2018-08-07 15:27:45
2      2018-08-08 08:38:49           2018-08-17 18:06:29


In [7]:
# Products - remove rows with missing dimensions (only 2 rows affected)
products.dropna(
    subset=[
        'product_weight_g',
        'product_length_cm',
        'product_height_cm',
        'product_width_cm'
    ],
    inplace=True
)

# Fill product metadata with sensible defaults
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# Reviews - fill missing text with placeholder (these are optional fields)
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No Title')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No Message')

# Payments - keep only valid rows (no negative values)
payments = payments[
    (payments['payment_value'] >= 0) &
    (payments['payment_installments'] >= 0)
]

# Order Items - keep only valid rows (price must be positive)
order_item = order_item[
    (order_item['price'] > 0) &
    (order_item['freight_value'] >= 0)
]

# Product dimensions validation - remove rows with zero/negative dimensions
products = products[
    (products['product_weight_g'] > 0) &
    (products['product_length_cm'] > 0) &
    (products['product_height_cm'] > 0) &
    (products['product_width_cm'] > 0)
]

# Filter delivered orders for later analysis
delivered_orders = orders[orders['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(delivered_orders)} out of {len(orders)} total")

Delivered orders: 96478 out of 99441 total


In [8]:
summary_data = []
for name, df in df_to_clean.items():
    summary_data.append({
        'Dataset': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Remaining Missing': df.isnull().sum().sum(),
        'Remaining Duplicates': df.duplicated().sum()
    })

quality_report_df = pd.DataFrame(summary_data)
print(quality_report_df.to_string(index=False))

             Dataset   Rows  Columns  Remaining Missing  Remaining Duplicates
           customers  99441        5                  0                     0
            products  32949        9                  0                     0
             reviews  99224        7                  0                     0
              orders  99441        8               4908                     0
category_translation     71        2                  0                     0
            location 738332        5                  0                     0
            payments 103886        5                  0                     0
             sellers   3095        4                  0                     0
          order_item 112650        7                  0                     0


### Step 3: Data Integration (Critical Component)

You must construct a **Master Dataset** by merging multiple tables.

Recommended sequence:

1. orders + customers
2. orders + order_items
3. order_items + products
4. orders + payments
5. orders + reviews
6. order_items + sellers
7. products + category_translation

**Final output:** A **single consolidated dataset** representing a unified business view

In [9]:
# Step-by-step merge to build the master dataset
# Each merge joins on a shared key column using left join to preserve all orders

#1. orders + customers (link each order to its customer)
orders_customers = pd.merge(orders, customers, on='customer_id', how='left')

#2. orders + order_items (link each order to items purchased)
orders_items = pd.merge(orders_customers, order_item, on='order_id', how='left')

#3. order_items + products (link each item to product details)
items_products = pd.merge(orders_items, products, on='product_id', how='left')

#4. products + category_translation (get English category names)
items_translated = pd.merge(items_products, category_translation, on='product_category_name', how='left')

#5. order_items + sellers (link items to their sellers)
items_sellers = pd.merge(items_translated, sellers, on='seller_id', how='left')

#6. orders + payments (link payment info)
items_payments = pd.merge(items_sellers, payments, on='order_id', how='left')

#7. orders + reviews (link customer reviews)
master_df = pd.merge(items_payments, reviews, on='order_id', how='left')

print(f"Master dataset shape: {master_df.shape}")
print(f"Total columns: {master_df.shape[1]}")

Master dataset shape: (119143, 40)
Total columns: 40


In [10]:
master_df.info()

master_df.shape

master_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119143 entries, 0 to 119142
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       119143 non-null  object        
 1   customer_id                    119143 non-null  object        
 2   order_status                   119143 non-null  object        
 3   order_purchase_timestamp       119143 non-null  datetime64[ns]
 4   order_approved_at              118966 non-null  datetime64[ns]
 5   order_delivered_carrier_date   117057 non-null  datetime64[ns]
 6   order_delivered_customer_date  115722 non-null  datetime64[ns]
 7   order_estimated_delivery_date  119143 non-null  datetime64[ns]
 8   customer_unique_id             119143 non-null  object        
 9   customer_zip_code_prefix       119143 non-null  int64         
 10  customer_city                  119143 non-null  object        
 11  

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,payment_sequential,payment_type,payment_installments,payment_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,credit_card,1.0,18.12,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,3.0,voucher,1.0,2.00,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,2.0,voucher,1.0,18.59,a54f0611adc9ed256b57ede6b6eb5114,4.0,No Title,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,boleto,1.0,141.46,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,credit_card,3.0,179.12,e73b67b67587f7644d5bd1a52deb1b01,5.0,No Title,No Message,2018-08-18,2018-08-22 19:07:58


Data integration - Building customer 360

Customer system
Order system
product system
Payment system
Review System

One review everything about customer + order + product + payments + reviews

In [11]:
customer_360 = master_df.groupby('customer_id').agg(
    #customer info
    customer_unique_id=('customer_unique_id', 'first'),
    customer_city=('customer_city', 'first'),
    customer_state=('customer_state', 'first'),
    #Order System
    total_orders=('order_id', 'nunique'),
    total_items_bought=('product_id', 'count'),
    first_purchase_date=('order_purchase_timestamp', 'min'),
    latest_purchase_date=('order_purchase_timestamp', 'max'),
    order_statuses=('order_status', lambda x: list(x.unique())),
    #Product System
    cateogries_purchased=('product_category_name_english', lambda x: list(x.dropna().unique())),
    distinct_products_bought=('product_id','nunique'),
    #Review
    average_review_score= ('review_score', 'mean'),
    total_reviews_submitted=('review_id', 'nunique'),
    latest_review_comment=('review_comment_message', lambda x: x.dropna().iloc[-1] if not x.dropna().empty else None)).reset_index()
    
print(customer_360.head())

                        customer_id                customer_unique_id  \
0  00012a2ce6f8dcda20d059ce98491703  248ffe10d632bebe4f7267f1f44844c9   
1  000161a058600d5901f007fab4c27140  b0015e09bb4b6e47c52844fab5fb6638   
2  0001fd6190edaaf884bcaf3d49edf079  94b11d37cd61cb2994a194d11f89682b   
3  0002414f95344307404f0ace7a26f1d5  4893ad4ea28b2c5b3ddf4e82e79db9e6   
4  000379cdec625522490c315e70c7a9fb  0b83f73b19c2019e182fd552c048a22c   

  customer_city customer_state  total_orders  total_items_bought  \
0        osasco             SP             1                   1   
1   itapecerica             MG             1                   1   
2  nova venecia             ES             1                   1   
3      mendonca             MG             1                   1   
4     sao paulo             SP             1                   1   

  first_purchase_date latest_purchase_date order_statuses  \
0 2017-11-14 16:08:26  2017-11-14 16:08:26    [delivered]   
1 2017-07-16 09:40:32  2017-07

### Step 4: Feature Engineering

Create meaningful features such as:

* Total order value (aggregated from order_items or payments)
* Delivery time (order purchase to delivery date)
* Number of items per order
* Customer purchase frequency
* Customer lifetime value (basic approximation)
* Average order value per customer

In [12]:
# Feature: Delivery time in days (difference between delivery and purchase)
master_df['delivery_time_days'] = (
    master_df['order_delivered_customer_date'] - master_df['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

# Order-level features: value and item count per order
order_features = master_df.groupby('order_id').agg(
    customer_id=('customer_id', 'first'),
    delivery_time_days=('delivery_time_days', 'first'),
    number_of_items_per_order=('product_id', 'count'),
    total_order_value=('payment_value', 'sum')
).reset_index()

print("Order features sample (delivery_time_days should now show real values):")
print(order_features.head())

Order features sample (delivery_time_days should now show real values):
                           order_id                       customer_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  3ce436f183e68e07877b285a838db11a   
1  00018f77f2f0320c557190d7a144bdd3  f6dd3ec061db4e3987629fe6b26e5cce   
2  000229ec398224ef6ca0657da4fc703e  6489ae5e4333f3693df5ad4372dab6d3   
3  00024acbcdf0a6daa1e931b038114c75  d4eb9395c8c0431ee92fce09860c5a06   
4  00042b26cf59d7ce69dfabb4e55b4fd9  58dbd0b2d70206bf40e62cd34e84d795   

   delivery_time_days  number_of_items_per_order  total_order_value  
0            7.614421                          1              72.19  
1           16.216181                          1             259.83  
2            7.948437                          1             216.87  
3            6.147269                          1              25.78  
4           25.114352                          1             218.04  


In [13]:
# Customer-level features (aggregated from order_features)

# Customer purchase frequency: how many orders each customer placed
customer_purchase_frequency = order_features.groupby('customer_id')['order_id'].count()

# Customer lifetime value (basic approximation): total spent across all orders
customer_lifetime_value = order_features.groupby('customer_id')['total_order_value'].sum()

# Average order value per customer
average_order_value_per_customer = order_features.groupby('customer_id')['total_order_value'].mean()

# Summary of engineered features
print("Customer-level feature summary:")
print(f"  Purchase frequency - mean: {customer_purchase_frequency.mean():.2f}, max: {customer_purchase_frequency.max()}")
print(f"  Lifetime value - mean: {customer_lifetime_value.mean():.2f}, max: {customer_lifetime_value.max():.2f}")
print(f"  Avg order value - mean: {average_order_value_per_customer.mean():.2f}")

Customer-level feature summary:
  Purchase frequency - mean: 1.00, max: 1
  Lifetime value - mean: 206.95, max: 109312.64
  Avg order value - mean: 206.95


### Step 5: Exploratory Data Analysis (EDA)

Perform structured analysis across the following dimensions:

#### Customer Analysis

* New vs repeat customers
* High-value vs low-value customers
* Geographic distribution of customers

#### Revenue and Order Analysis

* Monthly revenue trends
* Order volume trends
* Peak sales periods

#### Product Analysis

* Top-selling product categories
* Revenue contribution by category
* Product demand distribution

#### Seller Analysis

* Top-performing sellers
* Seller contribution to revenue
* Seller distribution

#### Review and Satisfaction Analysis

* Distribution of review scores
* Relationship between delivery time and ratings
* Identification of dissatisfaction patterns

### Step 6: Data Visualization

Use Matplotlib and Seaborn to create:

* Time series plots (sales trends)
* Bar charts (category performance)
* Histograms (distribution analysis)
* Box plots (outlier detection)
* Heatmaps (correlation analysis)

All visualizations must be clearly labeled and interpretable.

### Step 7: Business Insights and Recommendations

You must derive clear and actionable insights:

* Identify top revenue-driving factors
* Highlight customer behavior patterns
* Evaluate operational inefficiencies
* Provide strategic recommendations

Insights must be supported by data and visual evidence.